# IrResnet4: A/B на `dataset_v002` и сравнение с v003

Два прогона на **`dataset_v002`** с протоколом [`train_irresnet_original.yaml`](../configs/train_irresnet_original.yaml) (hidden=72, lr=1e-5, WRS, StepLR):

| Run | `label_schema` | Смысл меток |
|-----|----------------|-------------|
| `colab06_v002_smarts` | `structure_smarts` | SMARTS совпал (baseline на v002) |
| `colab06_v002_structure` | `structure` | SMARTS **и** пик в регионе |

Референс: `colab06_irresnet_dataset_v003` — `structure_smarts` на v003 (split 70/10/20).

**Drive:** `.../ir_data/processed/dataset_v002/` с `labels_structure.parquet` и `labels_structure_smarts.parquet`.

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path('IR_expert_system_3')
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/Lamblador/IR_expert_system_3.git', str(REPO_DIR)], check=True)
%cd IR_expert_system_3
!pip install -q -e ".[torch]" iterative-stratification

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
IR_DATA = Path('/content/drive/MyDrive/ir_data')
RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print('IR_DATA', IR_DATA.exists(), IR_DATA)
print('RUNS_DRIVE', RUNS_DRIVE)

In [ ]:
import os
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, resolve_paths

os.environ['IR_PROCESSED_ROOT'] = str(IR_DATA / 'processed')

paths_cfg = load_yaml(Path('configs/paths.colab.yaml'))
paths_cfg['dataset_version'] = 'dataset_v002'  # явно v002, не auto→v003
paths = resolve_paths(paths_cfg)
DATASET_DIR = paths['processed_root'] / paths['dataset_version']

assert (DATASET_DIR / 'spectra.npz').is_file(), f'Нет spectra.npz: {DATASET_DIR}'
assert (DATASET_DIR / 'labels_structure.parquet').is_file()
assert (DATASET_DIR / 'labels_structure_smarts.parquet').is_file()

split_path = DATASET_DIR / 'split.json'
if split_path.is_file():
    print('split.json (начало):', split_path.read_text(encoding='utf-8')[:300])
else:
    print('split.json отсутствует — выполните ячейку dataset-split ниже')
print('dataset:', DATASET_DIR)

## Split v2 на v002 (рекомендуется)

Без `val_ids` в split v1 код использует **test как val** — early stop и подбор порога нечестны относительно v003. Выполните один раз перед обучением (стратификация по `structure_smarts` достаточна для обоих прогонов).

In [ ]:
!ir-pipeline dataset-split \
  --paths configs/paths.colab.yaml \
  --dataset-version dataset_v002 \
  --label-schema structure_smarts

split_path = DATASET_DIR / 'split.json'
print(split_path.read_text(encoding='utf-8')[:400] if split_path.is_file() else 'split.json не создан')

In [ ]:
import pandas as pd
from ir_pipeline.dataset_preview import build_multilabel_matrix
from ir_pipeline.resnet_input import load_model_inputs

_, _, spec_ids, _, _ = load_model_inputs(DATASET_DIR)
bands = paths['bands_config']
rows = []
for schema in ['structure_smarts', 'structure', 'spectrum']:
    try:
        Y, _ = build_multilabel_matrix(DATASET_DIR, spec_ids, bands, label_schema=schema)
        rows.append({
            'schema': schema,
            'mean_labels': float(Y.sum(axis=1).mean()),
            'positives': int(Y.sum()),
        })
    except FileNotFoundError as e:
        rows.append({'schema': schema, 'error': str(e)})
pd.DataFrame(rows)

In [ ]:
%matplotlib inline
from copy import deepcopy
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import IrResnetTrainer

base_cfg = merge_train_defaults(load_yaml(Path('configs/train_irresnet_original.yaml')))

RUNS = [
    ('structure_smarts', 'colab06_v002_smarts'),
    ('structure', 'colab06_v002_structure'),
]

summaries = []
run_dirs = {}
train_cfgs = {}
for schema, run_suffix in RUNS:
    cfg = deepcopy(base_cfg)
    cfg['label_schema'] = schema
    run_dir = Path('runs') / run_suffix
    print('===', schema, '=>', run_dir)
    trainer = IrResnetTrainer(
        dataset_dir=DATASET_DIR,
        run_dir=run_dir,
        bands_yaml=paths['bands_config'],
        train_cfg=cfg,
        label_schema=schema,
    )
    summary = trainer.fit()
    summary['run_suffix'] = run_suffix
    summaries.append(summary)
    run_dirs[run_suffix] = run_dir
    train_cfgs[run_suffix] = cfg
    print(summary)

In [ ]:
import json
import matplotlib.pyplot as plt
from ir_pipeline.train_monitor import IrResnetTrainingPlotter

for run_suffix, run_dir in run_dirs.items():
    cfg = train_cfgs[run_suffix]
    plotter = IrResnetTrainingPlotter.from_train_cfg(run_dir, cfg, title=f'IrResnet4 — {run_suffix}')
    hist_path = run_dir / 'irresnet_history.json'
    if hist_path.is_file():
        plotter.series = json.loads(hist_path.read_text(encoding='utf-8'))
        plotter.live_plot = True
        plotter._clear_and_plot(
            len(plotter.series.get('train_loss', [])),
            int(cfg.get('torch_epochs', 200)),
        )
    else:
        img = run_dir / 'irresnet_training_curve.png'
        if img.is_file():
            plt.figure(figsize=(10, 4))
            plt.imshow(plt.imread(img))
            plt.title(run_suffix)
            plt.axis('off')
            plt.show()

In [ ]:
import shutil
from pathlib import Path

for run_suffix, run_dir in run_dirs.items():
    run_name = f'{run_suffix}_dataset_v002'
    dest = RUNS_DRIVE / run_name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(run_dir, dest)
    shutil.make_archive(str(RUNS_DRIVE / run_name), 'zip', run_dir)
    print('Saved to Drive:', dest)
    print('Zip:', (RUNS_DRIVE / run_name).with_suffix('.zip'))

## Сравнение с v003

Сводка по `irresnet_metrics.json`: v002 A/B + референс `colab06_irresnet_dataset_v003` (локально или на Drive).

In [ ]:
import json
import pandas as pd
from pathlib import Path

METRIC_KEYS = [
    'label_schema',
    'test_lrap',
    'test_f1_weighted',
    'test_f1_macro',
    'test_f1_micro',
    'val_lrap',
    'best_epoch',
]

def load_run_metrics(run_dir: Path) -> dict:
    p = run_dir / 'irresnet_metrics.json'
    if not p.is_file():
        return {'run': run_dir.name, 'error': 'metrics not found'}
    m = json.loads(p.read_text(encoding='utf-8'))
    row = {'run': run_dir.name}
    for k in METRIC_KEYS:
        row[k] = m.get(k)
    row['dataset'] = m.get('dataset_dir', str(run_dir))
    return row

# Локальные прогоны v002
compare_rows = [load_run_metrics(run_dirs[s]) for s in run_dirs]

# Референс v003: локально или на Drive
v003_candidates = [
    Path('runs/colab06_irresnet_dataset_v003'),
    RUNS_DRIVE / 'colab06_irresnet_dataset_v003',
    Path('../runs/colab06_irresnet_dataset_v003'),
]
for cand in v003_candidates:
    if (cand / 'irresnet_metrics.json').is_file():
        compare_rows.append(load_run_metrics(cand))
        break

pd.DataFrame(compare_rows)